# Model Service for Code Generation 

In this section, we demonstrate how to deploy a code generation service that provides a REST API endpoint for generating code based on natural language queries. This service provides a REST API endpoint for generating code based on natural language queries, 
The service includes performance optimization features to handle large repositories:
- **Parameter-based operation control** with `metadata_only` mode for fast processing 
- **Resource optimization** with batched processing and configurable timeouts

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Register and Log the Model to MLFlow
- API Usage Examples

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 41 ms, sys: 11 ms, total: 51.9 ms
Wall time: 1.8 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-16 17:05:09 - INFO - Notebook execution started.


# Install and Import Libraries

In [ ]:
# === Built-in ===
import os
import sys
import datetime
from pathlib import Path
from typing import List
import mlflow
import logging

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

# === Third-party libraries ===
import pandas as pd
import warnings
import yaml
from IPython import get_ipython
from IPython.display import display, Markdown, Code

# === Langchain ===
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

# === Internal modules ===
 
# Utils
from src.utils import (
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    initialize_llm,
    configure_hf_cache,
    clean_code,
    generate_code_with_retries,
    get_context_window,
    dynamic_retriever,
    format_docs_with_adaptive_context
)

# Import Logger from new MLflow structure
from src.mlflow import Logger

from IPython import get_ipython

# Configure Settings

In [6]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [7]:
# === Configuration settings ===
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
DEMO_FOLDER = "../demo"
MLFLOW_MODEL_NAME = "Code-Generation-Model"

# Define embedding model parameters - using industry-standard models/ directory
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# Store embedding model in models/embeddings/ following industry standards
EMBEDDING_MODEL_PATH = os.path.join("../models/embeddings", EMBEDDING_MODEL_NAME)

In [8]:
# Configure HuggingFace cache for the embedding model
configure_hf_cache()

## Configuration and Secrets Loading

In this section, we load configuration parameters and API keys from separate YAML files. This separation helps maintain security by keeping sensitive information (API keys) separate from configuration settings.

- **config.yaml**: Contains non-sensitive configuration parameters like model sources and URLs
- **secrets.yaml**: Contains sensitive API keys for services like HuggingFace
- *(Optional for Premium users)* Secrets such as API keys for services like HuggingFace can be stored as environment variables for the project and loaded into the notebook (see the project's README file for steps on how to save secrets in Secrets Manager).

In [9]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

No secrets file found at ../configs/secrets.yaml; relying on preexisting environment
✅ Configuration loaded successfully
✅ Secrets loaded successfully


# Register and Log the Model to MLFlow

## Model Architecture


### Directory Structure
```
models/
├── embeddings/           # Text embedding models for vector search
│   └── all-MiniLM-L6-v2/ # Sentence transformer model  
└── llm/                  # Large Language Models (if stored locally)

data/                     # Documents for RAG context
demo/                     # UI components and demos
src/mlflow/              # Universal MLflow integration
```

The Logger will automatically detect and package embedding models from `models/embeddings/` into the MLflow artifacts, while the Loader retrieves them for inference.

In [10]:
%%time

mlflow.set_tracking_uri('/phoenix/mlflow')
# Set up the MLflow experiment
mlflow.set_experiment(MLFLOW_MODEL_NAME)

# Get model path from config
model_path = config.get("model_path")

# Check if the model file exists when using local model source
if config.get("model_source") == "local" and model_path and os.path.exists(model_path):
    logger.info(f"✅ LLM model file found at {model_path}")
else:
    if config.get("model_source") == "local":
        logger.info(f"⚠️ Warning: LLM model file not found at {model_path}. Please verify the path.")
    else:
        logger.info(f"Using non-local model source: {config.get('model_source')}")
        model_path = None

# === Handle embedding model with industry-standard structure ===
# Store embedding models in models/embeddings/ directory following clean architecture
if not os.path.exists(EMBEDDING_MODEL_PATH):
    # Create the models/embeddings directory structure if it doesn't exist
    try:
        from sentence_transformers import SentenceTransformer
        os.makedirs(EMBEDDING_MODEL_PATH, exist_ok=True)
        logger.info(f"📁 Created embedding models directory: {EMBEDDING_MODEL_PATH}")
        logger.info(f"⬇️ Downloading and saving embedding model '{EMBEDDING_MODEL_NAME}'...")
        
        # Download and save the embedding model
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        model.save(EMBEDDING_MODEL_PATH)
        logger.info(f"✅ Embedding model saved successfully at {EMBEDDING_MODEL_PATH}")
        
    except Exception as e:
        logger.error(f"❌ Error saving embedding model: {str(e)}")
        logger.warning("⚠️ Will proceed without a local embedding model. The service will download it during initialization.")
        EMBEDDING_MODEL_PATH = None
else:
    logger.info(f"✅ Embedding model already exists at {EMBEDDING_MODEL_PATH}")

# === Define model input/output schema for code generation ===
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Define model input/output schema with all parameters
input_schema = Schema([
    ColSpec("string", "question"),
    ColSpec("string", "repository_url", required=False),  # Optional repository URL
    #ColSpec("boolean", "metadata_only", required=False),  # Optional metadata-only flag
])
output_schema = Schema([
    ColSpec("string", "result")
])

# Create signature  
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Use the Logger's log_model method to register the model in MLflow
with mlflow.start_run(run_name=MLFLOW_MODEL_NAME) as run:
    # Log and register the model
    Logger.log_model(
        signature=signature,
        artifact_path="code_generation_model",
        config_path=CONFIG_PATH,
        docs_path="../data",  # Use a data directory
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER
    )
    
    # Register the model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/code_generation_model"
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )
    
    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")
    logger.info(f"🏗️ Artifacts structure includes:")
    logger.info(f"   📂 models/embeddings/ → Embedding models (industry standard)")
    logger.info(f"   📂 models/llm/ → LLM models (if local)")
    logger.info(f"   📂 data/ → Documents for RAG")
    logger.info(f"   📂 demo/ → UI components")

2026/04/16 17:05:15 INFO mlflow.tracking.fluent: Experiment with name 'Code-Generation-Model' does not exist. Creating a new experiment.
2026-04-16 17:05:15 - INFO - ✅ LLM model file found at /home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
2026-04-16 17:05:18 - INFO - 📁 Created embedding models directory: ../models/embeddings/all-MiniLM-L6-v2
2026-04-16 17:05:18 - INFO - ⬇️ Downloading and saving embedding model 'all-MiniLM-L6-v2'...
2026-04-16 17:05:21 - INFO - ✅ Embedding model saved successfully at ../models/embeddings/all-MiniLM-L6-v2
Successfully registered model 'Code-Generation-Model'.
2026/04/16 17:10:11 WARNING mlflow.tracking._model_registry.fluent: Run with id d1a4ae646ebb44c88dfd7fe2c8fc53b5 has no artifacts at artifact path 'code_generation_model', registering model based on models:/m-78900ee78e3a4d2f88b69827e42441e7 instead
Created version '1' of model 'Code-Generation-Model'.
2026-04-16 17:10:12 - INFO - ✅ Model registered successfully wi

CPU times: user 6.89 s, sys: 32 s, total: 38.9 s
Wall time: 4min 56s


### API Usage Examples

The optimized code generation service can be invoked through its REST API using the following patterns:

#### Details

The API includes some features like:

1. **metadata_only mode**: Allows extracting only basic metadata without running the resource-intensive LLM analysis
2. **Timeout controls**: Prevents worker timeouts with configurable processing limits

```bash
# Example 1: Direct code generation (no repository)
curl -X 'POST' \
  'https://endpoint/invocations' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
  "inputs": {
    "question": "Write Python code to load the LLM model using Ollama with \'llama3\' and generate an inspirational quote."
  }
}'

# Example 2: Repository-enhanced code generation with metadata-only mode (FAST)
curl -X 'POST' \
  'https://endpoint/invocations' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
  "inputs": {
    "question": "Write Python code to scrape a website and extract all links",
    "repository_url": "https://github.com/MunGell/awesome-for-beginners"
  }
}'

# Example 3: Repository-enhanced code generation with full processing (DEEP UNDERSTANDING)
curl -X 'POST' \
  'https://endpoint/invocations' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{
  "inputs": {
    "question": "Write Python code to scrape a website and extract all links",
    "repository_url": "https://github.com/MunGell/awesome-for-beginners"
  }
}'
```

The service will automatically adapt to the parameters provided:
- With `metadata_only=true`, it performs lightweight processing ideal for large repositories
- With full processing, it performs deep analysis but take longer

In [11]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-16 17:10:12 - INFO - ⏱️ Total execution time: 5m 2.62s


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).